In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "maryclass29"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center([
    html.A(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '180px', 'height': 'auto'}
        ),
        href='https://www.snhu.edu',
        target='_blank'
    ),

    html.H1('Grazioso Salvare Rescue Dashboard'),

    html.H4('Created by Marielys Abraham Reyes')
]),
    
    # Interactive rescue filters
html.Div([
    html.H3("Find Your Search and Rescue Companion"),

    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
            {'label': 'Reset', 'value': 'Reset'}
        ],
        value='Reset',
        labelStyle={
            'display': 'inline-block',
            'marginRight': '20px'
        }
    )
], style={'textAlign': 'center', 'padding': '15px'}),

html.Hr(),

# Interactive animal data table
dash_table.DataTable(
    id='datatable-id',

    columns=[
        {"name": i, "id": i, "deletable": False, "selectable": True}
        for i in df.columns
    ],

    data=df.to_dict('records'),

    page_size=10,
    page_action='native',
    sort_action='native',
    filter_action='native',
    row_selectable='single',
    selected_rows=[0],
    selected_columns=[],

    style_table={'overflowX': 'auto'},

    style_cell={
        'textAlign': 'left',
        'padding': '10px',
        'minWidth': '120px',
        'maxWidth': '200px',
        'overflow': 'hidden',
        'textOverflow': 'ellipsis'
    },

    style_header={
        'backgroundColor': '#E8F0F2',
        'fontWeight': 'bold'
    }
),

html.Br(),
html.Hr(),
        
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
   # Display the pie chart and map side by side
 html.Div(
    children=[
        html.Div(
            id='graph-id',
            style={'width': '50%', 'minWidth': '0'}
        ),

        html.Div(
            id='map-id',
            style={'width': '50%', 'minWidth': '0'}
        )
    ],
    style={
        'display': 'flex',
        'width': '100%',
        'gap': '15px',
        'alignItems': 'flex-start'
    }
 )
        
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    # Show all animals when Reset is selected
    if filter_type == 'Reset':
        query = {}

    # Dogs suited for water rescue
    elif filter_type == 'Water':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': [
                'Labrador Retriever Mix',
                'Chesapeake Bay Retriever',
                'Newfoundland'
            ]},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }

    # Dogs suited for mountain or wilderness rescue
    elif filter_type == 'Mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': [
                'German Shepherd',
                'Alaskan Malamute',
                'Old English Sheepdog',
                'Siberian Husky',
                'Rottweiler'
            ]},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }

    # Dogs suited for disaster rescue or individual tracking
    elif filter_type == 'Disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': [
                'Doberman Pinscher',
                'German Shepherd',
                'Golden Retriever',
                'Bloodhound',
                'Rottweiler'
            ]},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }

    else:
        query = {}

    # Retrieve matching records using the CRUD module
    filtered_df = pd.DataFrame.from_records(db.read(query))

    # Remove MongoDB's ObjectId before displaying the records
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')
    
# Update the breed distribution chart when the table changes
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    # Use the currently displayed animal records
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)

    # Display a message if no animals match the filters
    if dff.empty:
        return html.P("No animals found for this selection.")

    # Count the number of animals in each breed
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['Breed', 'Count']

    # Create an interactive pie chart
    fig = px.pie(
        breed_counts,
        names='Breed',
        values='Count',
        title='Animal Breed Distribution'
    )

    fig.update_layout(
        margin=dict(l=20, r=20, t=60, b=20),
        legend_title_text='Breeds'
    )

    return [
        dcc.Graph(
            figure=fig,
            style={'width': '100%', 'height': '500px'}
        )
    ]
    
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
# Update the map based on the filtered animal records
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'data')]
)
def update_map(viewData):

    # Get the records currently displayed in the table
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)

    # Check if there are animals to display
    if dff.empty:
        return html.P("No animal locations available.")

    # Keep only animals with valid coordinates
    dff['location_lat'] = pd.to_numeric(
        dff['location_lat'], errors='coerce'
    )
    dff['location_long'] = pd.to_numeric(
        dff['location_long'], errors='coerce'
    )

    dff = dff.dropna(
        subset=['location_lat', 'location_long']
    )

    if dff.empty:
        return html.P("No valid animal locations found.")

    # Display the first animal with a valid location
    animal = dff.iloc[0]

    latitude = float(animal['location_lat'])
    longitude = float(animal['location_long'])

    return dl.Map(
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(str(animal['breed'])),
                    dl.Popup([
                        html.H4("Animal Information"),
                        html.P("Name: " + str(animal['name'])),
                        html.P("Breed: " + str(animal['breed']))
                    ])
                ]
            )
        ],
        center=[latitude, longitude],
        zoom=10,
        style={
            'width': '100%',
            'height': '500px'
        }
    )   


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://profilejazz-relaxscholar-3000.codio.io/proxy/8050/
